# FLUX-Net: Brain Tumor Classification (Kaggle T4)

**F**requency **L**ightweight **U**nified **X**-attention **Net**

**Setup:**
1. Upload `models.zip` as Kaggle Dataset named **fluxnet-models**
2. Add this notebook + `brain-tumor-mri-dataset` (masoudnickparvar) as inputs
3. Run all cells

In [1]:
# ============================================================
# PATH CONFIGURATION — edit paths if dataset names differ
# ============================================================
DATA_PATH = "/kaggle/input/datasets/lucky02655/braintumor-dataset/data/kaggle-7023"
OUTPUT_DIR = "/kaggle/working"
EXPERIMENT_NAME = "flux_kaggle_v1"
RESUME_PATH = None  # set to checkpoint path to resume

# Training hyperparams
SEED = 42
BATCH_SIZE = 16
EPOCHS = 200
LEARNING_RATE = 5e-4
NUM_WORKERS = 2
USE_AMP = True
N_FOLDS = 5
CURRENT_FOLD = 0
EARLY_STOPPING_PATIENCE = 25
LABEL_SMOOTHING = 0.1
DROPOUT = 0.3
# ============================================================

In [2]:
import sys, os, zipfile
import torch
import random
import numpy as np

# ---- Extract models.zip from Kaggle input datasets ----
ZIP_PATH = None
input_dir = "/kaggle/input"
if os.path.exists(input_dir):
    for d in os.listdir(input_dir):
        dp = os.path.join(input_dir, d)
        zip_candidate = os.path.join(dp, d + ".zip")
        if os.path.exists(zip_candidate):
            ZIP_PATH = zip_candidate
            break

if ZIP_PATH and os.path.exists(ZIP_PATH):
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall("/kaggle/working/")
    print(f"Extracted models from: {ZIP_PATH}")
else:
    print("No models.zip found. Place models/ package manually.")
MODEL_PATH = '/kaggle/input/models/lucky02655/braintumor/tensorflow2/default/1/models'
sys.path.insert(0, MODEL_PATH)


MODEL_ROOT = "/kaggle/input/models/lucky02655/braintumor/tensorflow2/default/1"

if MODEL_ROOT not in sys.path:
    sys.path.insert(0, MODEL_ROOT)
print("Model files:", sorted(f for f in __import__('os').listdir(MODEL_PATH) if f.endswith(".py")))
# --------------------------------------------------------

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision('high')
os.environ["PYTHONHASHSEED"] = str(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA: {torch.version.cuda}")
    USE_AMP = USE_AMP and True

No models.zip found. Place models/ package manually.
Model files: ['__init__.py', 'blocks.py', 'config.py', 'data.py', 'flux_net.py', 'fusion.py', 'head.py', 'losses.py', 'metrics.py', 'scheduler.py', 'ssm.py', 'trainer.py', 'utils.py']
Device: cuda
GPU: Tesla T4
CUDA: 12.8


In [3]:
import os

for root, dirs, files in os.walk("/kaggle/input/models/lucky02655/braintumor/pytorch/default/1"):
    print(root)

    for f in files:
        print("   ", f)

In [4]:
from models.config import Config, FLUXConfig, TrainingConfig, DataConfig

config = Config()
config.model = FLUXConfig(
    dims=(48, 96, 192, 320),
    depths=(2, 3, 6, 2),
    expansion=3,
    spectral_fusion=True,
    dropout=DROPOUT,
    num_classes=4,
    ssm_d_state=8,
    msap_proj_dim=128,
)
config.training = TrainingConfig(
    optimizer="adamw",
    lr=LEARNING_RATE,
    weight_decay=0.05,
    betas=(0.9, 0.999),
    scheduler="cosine_warm_restarts",
    warmup_epochs=10,
    T_0=30,
    T_mult=2,
    eta_min=1e-6,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    grad_accumulation_steps=8,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    use_amp=USE_AMP,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    early_stopping_min_delta=1e-4,
    label_smoothing=LABEL_SMOOTHING,
    n_folds=N_FOLDS,
    grad_clip=1.0,
    ema_decay=0.99,
    fold=CURRENT_FOLD,
    seed=SEED,
    save_best=True,
    save_last=True,
)
config.data = DataConfig(
    data_root=DATA_PATH,
    dataset_type="kaggle7023",
    kaggle_train_dir="Training",
    kaggle_test_dir="Testing",
    img_size=224,
    class_names=("glioma", "meningioma", "notumor", "pituitary"),
    mean=(0.5, 0.5, 0.5),
    std=(0.5, 0.5, 0.5),
)
config.output_dir = OUTPUT_DIR
config.experiment_name = EXPERIMENT_NAME

os.makedirs(config.checkpoint_dir, exist_ok=True)
os.makedirs(config.log_dir, exist_ok=True)
os.makedirs(config.metrics_dir, exist_ok=True)

print(f"Data root: {config.data.data_root}")
print(f"Checkpoints: {config.checkpoint_dir}")
print(f"Logs: {config.log_dir}")
print(f"Metrics: {config.metrics_dir}")
print(f"Batch size: {config.training.batch_size}")
print(f"Epochs: {config.training.epochs}")
print(f"AMP: {config.training.use_amp}")
print(f"Dropout: {config.model.dropout}")
print(f"Label smoothing: {config.training.label_smoothing}")
print(f"Early stopping patience: {config.training.early_stopping_patience}")

Data root: /kaggle/input/datasets/lucky02655/braintumor-dataset/data/kaggle-7023
Checkpoints: outputs/flux_v2/checkpoints
Logs: outputs/flux_v2/logs
Metrics: outputs/flux_v2/metrics
Batch size: 16
Epochs: 200
AMP: True
Dropout: 0.3
Label smoothing: 0.1
Early stopping patience: 25


In [5]:
from models.trainer import Trainer

trainer = Trainer(config, device, resume_path=RESUME_PATH)
print(f"Train {len(trainer.train_loader)} batches | "
      f"Val {len(trainer.val_loader)} batches | "
      f"Test {len(trainer.test_loader) if trainer.test_loader else 0} batches")
print(f"FLUX-Net params: {sum(p.numel() for p in trainer.model.parameters())/1e6:.2f}M")

/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/kaggle/input/models/lucky02655/braintumor/tensorflow2/default/1/models/data.py:94: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 30.0), p=1.0),


Dataset: 5600 samples, {'glioma': 1400, 'meningioma': 1400, 'notumor': 1400, 'pituitary': 1400}
Fold 0/5: Train=4480, Val=1120
Test: 1600 samples
FLUX-Net v2 | Params: 8,505,168 total, 8,505,168 trainable
Train 280 batches | Val 35 batches | Test 50 batches
FLUX-Net params: 8.51M


In [6]:
best_metrics = trainer.train()


Training FLUX-Net v2 for 200 epochs (280 batches/epoch)
LR=5.00e-04  WD=0.05  Dropout=0.3  GradClip=1.0
Device: cuda  AMP: True  EMA: 0.99

Epoch 1/200  LR=5.00e-05  Best: 0.0000 @ ep0
  E1 [1/280]  loss=1.3864  acc=0.3125  gnorm=nan
  E1 [57/280]  loss=1.3861  acc=0.2708  gnorm=0.202
  E1 [113/280]  loss=1.3857  acc=0.3086  gnorm=0.541
  E1 [169/280]  loss=1.3852  acc=0.3376  gnorm=0.343
  E1 [225/280]  loss=1.3846  acc=0.3483  gnorm=0.694
  E1 [280/280]  loss=1.3839  acc=0.3679  gnorm=0.421

  ────────────────────────────────────────────────────
  Train — Loss: 1.3839  Acc: 0.3679  F1: 0.3355
  Val   — Loss: 1.3856  Acc: 0.3920  F1: 0.2612  AUC: 0.7923
  LR: 0.000100  Gap — Acc: -0.0241  Loss: +0.0018
  ────────────────────────────────────────────────────
    glioma       | train_f1=0.4286  val_f1=0.5115
    meningioma   | train_f1=0.2362  val_f1=0.0000
    notumor      | train_f1=0.2266  val_f1=0.0000
    pituitary    | train_f1=0.4506  val_f1=0.5335
  ★ Best model saved (val_acc=0

In [7]:
# Save final model weights
torch.save(trainer.model.state_dict(), 
           os.path.join(OUTPUT_DIR, "fluxnet_final.pth"))
print(f"Saved: {os.path.join(OUTPUT_DIR, 'fluxnet_final.pth')}")

Saved: /kaggle/working/fluxnet_final.pth


In [ ]:
print("\nBest validation results:")
for k, v in best_metrics.items():
    print(f"  {k}: {v}")

In [30]:
print("hi")

hi


In [34]:
!du -sh /kaggle/working/*

33M	/kaggle/working/fluxnet_final.pth
13G	/kaggle/working/outputs


In [32]:
import os

os.remove("/kaggle/working/kaggle_output.zip")
print("Deleted incomplete ZIP")

Deleted incomplete ZIP


In [36]:
!du -sh /kaggle/working/outputs

13G	/kaggle/working/outputs


In [39]:
!find /kaggle/working/outputs -maxdepth 1 -type d

/kaggle/working/outputs
/kaggle/working/outputs/.ipynb_checkpoints
/kaggle/working/outputs/flux_v2


In [40]:
!du -sh /kaggle/working/outputs/flux_v2/*

13G	/kaggle/working/outputs/flux_v2/checkpoints
536K	/kaggle/working/outputs/flux_v2/logs
8.0K	/kaggle/working/outputs/flux_v2/metrics


In [41]:
!ls -lh /kaggle/working/outputs/flux_v2/checkpoints

total 13G
-rw-r--r-- 1 root root 131M Jul  4 15:30 best_model.pt
-rw-r--r-- 1 root root 131M Jul  4 11:28 epoch_0.pt
-rw-r--r-- 1 root root 131M Jul  4 13:47 epoch_105.pt
-rw-r--r-- 1 root root 131M Jul  4 13:52 epoch_109.pt
-rw-r--r-- 1 root root 131M Jul  4 14:01 epoch_116.pt
-rw-r--r-- 1 root root 131M Jul  4 14:03 epoch_117.pt
-rw-r--r-- 1 root root 131M Jul  4 14:05 epoch_119.pt
-rw-r--r-- 1 root root 131M Jul  4 11:43 epoch_11.pt
-rw-r--r-- 1 root root 131M Jul  4 14:08 epoch_121.pt
-rw-r--r-- 1 root root 131M Jul  4 14:17 epoch_128.pt
-rw-r--r-- 1 root root 131M Jul  4 14:18 epoch_129.pt
-rw-r--r-- 1 root root 131M Jul  4 11:44 epoch_12.pt
-rw-r--r-- 1 root root 131M Jul  4 14:20 epoch_130.pt
-rw-r--r-- 1 root root 131M Jul  4 14:30 epoch_138.pt
-rw-r--r-- 1 root root 131M Jul  4 14:32 epoch_139.pt
-rw-r--r-- 1 root root 131M Jul  4 11:45 epoch_13.pt
-rw-r--r-- 1 root root 131M Jul  4 14:45 epoch_149.pt
-rw-r--r-- 1 root root 131M Jul  4 14:57 epoch_158.pt
-rw-r--r-- 1 root root

In [42]:
import os
import zipfile

base_dir = "/kaggle/working/outputs/flux_v2/checkpoints"
out_dir = "/kaggle/working"

# Files you want
core_files = ["best_model.pt", "last_model.pt"]

epoch_files = [
    "epoch_30.pt",
    "epoch_60.pt",
    "epoch_99.pt",
    "epoch_158.pt",
    "epoch_177.pt",
    "epoch_189.pt"
]

def make_zip(zip_name, file_list):
    zip_path = os.path.join(out_dir, zip_name)

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in file_list:
            f_path = os.path.join(base_dir, f)

            if os.path.exists(f_path):
                zf.write(f_path, arcname=f)
                print(f"Added: {f}")
            else:
                print(f"Missing (skipped): {f}")

    print("\nCreated:", zip_path)


# Create ZIP 1: best + last
make_zip("core_models.zip", core_files)

# Create ZIP 2: selected epochs
make_zip("epoch_samples.zip", epoch_files)

Added: best_model.pt
Added: last_model.pt

Created: /kaggle/working/core_models.zip
Added: epoch_30.pt
Added: epoch_60.pt
Added: epoch_99.pt
Added: epoch_158.pt
Added: epoch_177.pt
Added: epoch_189.pt

Created: /kaggle/working/epoch_samples.zip


In [43]:
import shutil
import os

src_dir = "/kaggle/working/outputs/flux_v2/checkpoints"
dst_dir = "/kaggle/working"

files_to_copy = [
    "best_model.pt",
    "last_model.pt"
]

for f in files_to_copy:
    src = os.path.join(src_dir, f)
    dst = os.path.join(dst_dir, f)

    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"Copied: {f} → /kaggle/working/")
    else:
        print(f"Missing: {f}")

print("\nDone. Check Kaggle 'Files' panel to download.")

Copied: best_model.pt → /kaggle/working/
Copied: last_model.pt → /kaggle/working/

Done. Check Kaggle 'Files' panel to download.
